In [1]:
import os

import napari
import numpy as np
import pandas as pd
from skimage import io
from ultralytics import YOLO


In [2]:
viewer = napari.Viewer()

In [ ]:
root = r'Z:\Collaborations\aif-botton-collaboration\Johannes\20250714\20250714_selectedPos_3_FocusPlane(order1).pos_1'
n_images = 20

chosen_indeces = np.unique(np.random.randint(0, len(os.listdir(root)), n_images))
files_to_chose = np.array(os.listdir(root))

viewer.layers.clear()
for idx, file in enumerate(files_to_chose):
    layer = viewer.open(os.path.join(root, file), plugin='napari-autoscape', downsample=16)[0]

    if idx <= 144:
        continue

Exception ignored in: <function WeakValueDictionary.__init__.<locals>.remove at 0x0000027334A8B560>
Traceback (most recent call last):
  File "c:\Users\aifadmin\mambaforge\envs\napari-autoscape\Lib\weakref.py", line 106, in remove
    self = selfref()
           ^^^^^^^^^
KeyboardInterrupt: 


In [ ]:
model = YOLO(r'C:\Users\aifadmin\Documents\Github\autoSCAPE\docs\notebooks\20250712_runs\runs\detect\train22\weights\best.pt')

In [5]:
rectangles = []
features = pd.DataFrame()

for layer in viewer.layers:
    io.imsave('image.png', layer.data.squeeze())
    boxes = model('image.png')[0].boxes

    for box, conf, cls in zip(boxes.xywhn.cpu().detach().numpy(), boxes.conf.cpu().detach().numpy(), boxes.cls.detach().numpy(), strict=False):

        z0 = layer.translate[0]
        y0 = layer.translate[1]
        x0 = layer.translate[2]

        w0 = layer.data.shape[2] * layer.scale[2]
        h0 = layer.data.shape[1] * layer.scale[1]

        xb = box[0]
        yb = box[1]
        wb = box[2]
        hb = box[3]

        x_L = x0 + (xb - wb / 2) * w0
        x_R = x0 + (xb + wb / 2) * w0

        y_T = y0 + (yb - hb / 2) * h0
        y_B = y0 + (yb + hb / 2) * h0


        rectangle = np.array([
            [z0, y_T, x_L],
            [z0, y_T, x_R],
            [z0, y_B, x_R],
            [z0, y_B, x_L]
        ])
        rectangles.append(rectangle)

        _features = pd.DataFrame(
            {
                'x': [x0 + xb * w0],
                'y': [y0 + yb * h0],
                'width_box': [wb * w0],
                'heigth_box': [hb * h0],
                'confidence': [conf],
                'class': [cls],
                'area': [wb * w0 * hb * h0]
             }
        )
        features = pd.concat([features, _features])


image 1/1 c:\Users\aifadmin\Documents\Github\autoSCAPE\docs\notebooks\image.png: 512x512 4 spheroids, 126.6ms
Speed: 4.6ms preprocess, 126.6ms inference, 1.8ms postprocess per image at shape (1, 3, 512, 512)

image 1/1 c:\Users\aifadmin\Documents\Github\autoSCAPE\docs\notebooks\image.png: 512x512 1 cell, 73.9ms
Speed: 5.0ms preprocess, 73.9ms inference, 1.9ms postprocess per image at shape (1, 3, 512, 512)

image 1/1 c:\Users\aifadmin\Documents\Github\autoSCAPE\docs\notebooks\image.png: 512x512 (no detections), 74.3ms
Speed: 3.0ms preprocess, 74.3ms inference, 1.5ms postprocess per image at shape (1, 3, 512, 512)

image 1/1 c:\Users\aifadmin\Documents\Github\autoSCAPE\docs\notebooks\image.png: 512x512 1 cell, 1 spheroid, 71.7ms
Speed: 3.2ms preprocess, 71.7ms inference, 1.9ms postprocess per image at shape (1, 3, 512, 512)

image 1/1 c:\Users\aifadmin\Documents\Github\autoSCAPE\docs\notebooks\image.png: 512x512 3 spheroids, 70.1ms
Speed: 3.0ms preprocess, 70.1ms inference, 1.9ms postp

In [6]:
shape_layer = viewer.add_shapes(rectangles, shape_type='rectangle', edge_color='class', edge_width=3, features=features)

In [7]:
face_colors = shape_layer.face_color
face_colors[:, -1] = 0
shape_layer.face_color = face_colors

c:\Users\aifadmin\mambaforge\envs\napari-autoscape\Lib\site-packages\biaplotter\colormap.py:34: UserWarning: Categorical colormap detected. Setting categorical=True. If the colormap is continuous, set categorical=False explicitly.
  warnings.warn(
c:\Users\aifadmin\mambaforge\envs\napari-autoscape\Lib\site-packages\biaplotter\artists.py:325: UserWarning: Color indices must be integers for categorical colormap. Change `overlay_colormap` to a continuous colormap or set `color_indices` to integers.
  warnings.warn(
c:\Users\aifadmin\mambaforge\envs\napari-autoscape\Lib\site-packages\biaplotter\artists.py:325: UserWarning: Color indices must be integers for categorical colormap. Change `overlay_colormap` to a continuous colormap or set `color_indices` to integers.
  warnings.warn(
c:\Users\aifadmin\mambaforge\envs\napari-autoscape\Lib\site-packages\biaplotter\colormap.py:34: UserWarning: Categorical colormap detected. Setting categorical=True. If the colormap is continuous, set categorical

In [10]:
viewer.camera.angles = (0,0,90)

In [ ]:
well_grid = viewer.layers['Well grid']
image_fused = viewer.layers['fused :: default_channel']

for idx, shape in enumerate(well_grid.data):

    row = well_grid.features['row'][idx]
    col = well_grid.features['col'][idx]

    # shape in pixel coordinates
    shape_px = (shape - image_fused.translate) / image_fused.scale
    limits = np.stack([
        np.min(shape_px, axis=0),
        np.max(shape_px, axis=0)
    ]).astype(int)[:, 1:]

    crop = image_fused.data[0][0,
        limits[0, 0] : limits[1, 0],
        limits[0, 1] : limits[1, 1],
    ]

    # translate_crop = image_fused.translate[1:] + limits[0, :] * image_fused.scale[1:]
    # layer = viewer.add_image(crop,
    #      translate=translate_crop,
    #      scale=image_fused.scale[1:],
    #      name='crop'
    #      )
    # layer.bounding_box.visible = True
    # break


In [47]:
row

0

In [40]:
well_grid.features

,row,column
0,0,0
1,0,1
2,0,2
3,0,3
4,0,4
...,...,...
891,31,23
892,31,24
893,31,25
894,31,26


In [12]:
image_fused.translate

array([    0.        ,  3950.60791148, 30328.39341148])

In [13]:
image_fused.scale

array([1.        , 0.32430006, 0.32430006])

In [16]:
(shape - image_fused.translate) / image_fused.scale

array([[    0.        ,  1514.38826377, 21570.54542731],
       [    0.        ,  1247.34379201, 21724.72362731],
       [    0.        ,   980.29932025, 21570.54542731],
       [    0.        ,   980.29932025, 21262.18902731],
       [    0.        ,  1247.34379201, 21108.01082731],
       [    0.        ,  1514.38826377, 21262.18902731]])